In [ ]:
!pip install torchio

In [ ]:
!pip install SimpleITK

In [ ]:
!pip install k3d

In [ ]:
import k3d
import pandas as pd
import numpy as np
import os
import glob
import SimpleITK as sitk
import json
import matplotlib.pyplot as plt
from PIL import Image
import torchio as tio
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split
from core.constants import seg_with_loc, organ2label
from core.preprocessing import resample_img, preprocess_img, image_clipping
from core.visualization import draw_img
from core.helpers import get_physical_point_and_location, get_idx

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
prefix = '/content/drive/Shareddrives/DATA/rsna-intracranial-aneurysm-detection/'
series_path = os.path.join(prefix, 'series')
segmentations_path = os.path.join(prefix, 'segmentations')

In [ ]:
localizers = pd.read_csv(os.path.join(prefix,'train_localizers.csv'))
train_df = pd.read_csv(os.path.join(prefix, 'train.csv'))

In [ ]:
localizers.columns

In [ ]:
train_folder = Path('/content/drive/Shareddrives/DATA/g143_review/D_Szyjka/images_and_labels/train')
test_folder = Path('/content/drive/Shareddrives/DATA/g143_review/D_Szyjka/images_and_labels/test')
train_folder.mkdir(exist_ok=True)
test_folder.mkdir(exist_ok=True)

In [ ]:
more_then_1_loc = localizers.groupby('SeriesInstanceUID')['location'].nunique()
more_then_1_loc = more_then_1_loc[more_then_1_loc > 1]
more_then_1_loc

In [ ]:
cta_x, mr_x, cta_y, mr_y = [], [], [], []

for i, ser in enumerate(seg_with_loc):
    if i % 10 == 0:
        print(i)

    ser_modality = train_df.loc[train_df['SeriesInstanceUID'] == ser]['Modality'].iloc[0]

    if ser_modality == 'CTA':
        x_lst = cta_x
        y_lst = cta_y
    else:
        x_lst = mr_x
        y_lst = mr_y

    seg_path = os.path.join(segmentations_path, f'{ser}_cowseg.nii')
    seg = sitk.ReadImage(seg_path)
    seg = sitk.DICOMOrient(seg)
    seg = resample_img(seg, is_mask=True)
    seg_arr = sitk.GetArrayFromImage(seg)

    nii_path = os.path.join(segmentations_path, f'{ser}.nii')
    nii_img = sitk.ReadImage(nii_path)
    nii_img = sitk.DICOMOrient(nii_img)
    nii_img = resample_img(nii_img)
    nii_arr = sitk.GetArrayFromImage(nii_img)

    sops = localizers.loc[localizers['SeriesInstanceUID'] == ser]

    ser_i = []

    for idx, row in sops.iterrows():
        slice_path = os.path.join(series_path, row.SeriesInstanceUID, f'{row.SOPInstanceUID}.dcm')
        dcm_img = sitk.ReadImage(slice_path)

        x, y, loc = get_physical_point_and_location(row)
        x, y, z = np.array(dcm_img.TransformContinuousIndexToPhysicalPoint((x, y, 0)))
        label = organ2label[loc]

        nii_k, nii_j, nii_i = nii_img.TransformPhysicalPointToIndex((x, y, z))
        ser_i.append(nii_i)

        stacked = nii_arr[nii_i - 2 : nii_i + 3, :, :]
        seg_extract = seg_arr[nii_i - 2 : nii_i + 3, :, :]

        mask = preprocess_img(seg_extract)
        mask = np.where(mask > 0, mask, 0)

        stacked = preprocess_img(stacked)
        stacked = image_clipping(stacked, ser_modality)
        stacked = np.concatenate((stacked, mask), axis=-1)

        x_lst.append(stacked)
        y_lst.append(label)

    min_i = min(ser_i)
    max_i = max(ser_i)

    # no aneurysm
    try:
        under_aneurysm = nii_arr[min_i - 9 : min_i - 4, :, :]
        under_aneurysm = preprocess_img(under_aneurysm)
        under_aneurysm = image_clipping(under_aneurysm, ser_modality)

        mask = seg_arr[min_i - 9 : min_i - 4, :, :]
        mask = preprocess_img(mask)
        mask = np.where(mask > 0, mask, 0)

        under_aneurysm = np.concatenate((under_aneurysm, mask), axis=-1)

        x_lst.append(under_aneurysm)
        y_lst.append(0)

    except IndexError:
        pass

    try:
        over_aneurysm = nii_arr[max_i + 5 : max_i + 10, :, :]
        over_aneurysm = preprocess_img(over_aneurysm)
        over_aneurysm = image_clipping(over_aneurysm, ser_modality)

        mask = seg_arr[max_i + 5 : max_i + 10, :, :]
        mask = preprocess_img(mask)
        mask = np.where(mask > 0, mask, 0)

        over_aneurysm = np.concatenate((over_aneurysm, mask), axis=-1)

        x_lst.append(over_aneurysm)
        y_lst.append(0)

    except IndexError:
        pass

cta_x = np.stack(cta_x, axis=0)
mr_x = np.stack(mr_x, axis=0)
cta_y = np.array(cta_y)
mr_y = np.array(mr_y)

Manual data review


In [ ]:
idx_gen = get_idx()

In [ ]:
cta_idx = next(idx_gen)

while cta_idx < cta_y.shape[0] and cta_y[cta_idx] > 0:
    cta_idx = next(idx_gen)

if cta_idx < cta_y.shape[0] and cta_y[cta_idx] == 0:
    draw_img(cta_x[cta_idx])
    print(f'class: {cta_y[cta_idx]}')
    print(f'index: {cta_idx}')

else:
    print('CTA removing done')

CHANGED OR REMOVED DATA INDICES IN CTA IMAGES:

1.  index 17 - Still aneurysm ispresent -> changed to 13
2. index 67 - No circle of willis  -> removed
3. index 71 - No circle of willis  -> removed
4. index 69 - Invalid data -> removed
5. index 68 - Invalid data -> removed
6. index 74 - No circle of willis -> removed
7. index 82 - Still aneurysm present -> changed to 10
8. index 83 - Still aneurysm present -> changed to 10
9. index 114 - Still aneurysm present -> changed to 13
10. index 115 - not sure it's not aneurysm -> removed
11. index 118 - not sure it's not aneurysm -> removed
12. index 139 - still aneurysm is present -> changed to 1
13. index 140 - still aneurysm is present -> changed to 1
14. index 195 - no circle of willis -> removed
15. index 217 - still aneurysm is present -> changed to 5
16. index 218 - still aneurysm is present -> changed to 5

In [ ]:
cta_y[[17, 114]] = 13
cta_y[[82, 83]] = 10
cta_y[[139, 140]] = 1
cta_y[[217, 218]] = 5
cta_x = np.delete(cta_x, [67, 71, 69, 68, 74, 115, 118, 195], axis=0)
cta_y = np.delete(cta_y, [67, 71, 69, 68, 74, 115, 118, 195])

In [ ]:
draw_img(cta_x[26])
cta_y[26]

In [ ]:
idx_gen = get_idx()

In [ ]:
mr_idx = next(idx_gen)

while mr_idx < mr_y.shape[0] and mr_y[mr_idx] > 0:
    mr_idx = next(idx_gen)

if mr_idx < mr_y.shape[0] and mr_y[mr_idx] == 0:
    draw_img(mr_x[mr_idx])
    print(f'class: {mr_y[mr_idx]}')
    print(f'index: {mr_idx}')

else:
    print('MR removing done')

CHANGED OR REMOVED DATA INDICES IN MR IMAGES:


1. index 26 - aneurysm's present -> changed to 1
2. index 27 - aneurysm's present -> changed to 1
3. index 29 - aneurysm's present -> changed to 1
4. index 30 - aneurysm's present -> changed to 1
5. index 38 - not sure it's aneurysm or not -> removed
6. index 39 - aneurysm's present -> changed to 1
7. index 64 - no circle of willis -> removed
8. index 73 - black image -> removed
9. index 156 - no circle of willis -> removed
10. index 157 - no circle of willis -> removed
11. index 176 - black image -> removed
12. index 192 - no circle of willis -> removed
13. index 194 - aneurysm's present -> changed to 10
14. index 195 - aneurysm's present -> changed to 10
15. index 197 - no circle of willis -> removed
16. index 212 - no circle of willis -> removed
17. index 225 - black image -> removed
18. index 244 - no circle of willis -> removed

In [ ]:
draw_img(mr_x[24])
mr_y[24]

In [ ]:
mr_y[[26, 27, 29, 30, 39]] = 1
mr_y[[194, 195]] = 10
mr_x = np.delete(mr_x, [38, 64, 73, 156, 157, 176, 192, 197, 212, 225, 244], axis=0)
mr_y = np.delete(mr_y, [38, 64, 73, 156, 157, 176, 192, 197, 212, 225, 244])

In [ ]:
y12_idx = np.where(cta_y == 12)[0]
x12 = cta_x[y12_idx, :]
y12 = cta_y[y12_idx]
cta_x = np.delete(cta_x, y12_idx, axis=0)
cta_y = np.delete(cta_y, y12_idx)
y12 = np.array([y12, np.array([1])]).T

In [ ]:
cta_y = np.column_stack((cta_y, np.ones(cta_y.shape)))
mr_y = np.column_stack((mr_y, np.zeros(mr_y.shape)))

x = np.vstack((cta_x, mr_x))
y = np.vstack((cta_y, mr_y))

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y[:,0], random_state=42)
x_train = np.concatenate((x_train, x12), axis=0)
y_train = np.concatenate((y_train, y12), axis=0)

In [ ]:
np.unique(x_train[:, :, :, 5:], return_counts=True)

In [ ]:
train_cta_mask = y_train[:,1] == 1
test_cta_mask = y_test[:, 1] == 1
test_mr_mask = y_test[:, 1] == 0
train_mr_mask = y_train[:, 1] == 0


x_train[train_cta_mask, :, :, :5], x_test[test_cta_mask, :, :, :5] = z_score(
    x_train[train_cta_mask, :, :, :5],
    x_test[test_cta_mask, :, :, :5])

x_train[train_mr_mask, :, :, :5], x_test[test_mr_mask, :, :, :5] = z_score(
    x_train[train_mr_mask, :, :, :5],
    x_test[test_mr_mask, :, :, :5])

y_test = np.delete(y_test, 1, axis=1)
y_train = np.delete(y_train, 1, axis=1)

In [ ]:
np.unique(x_train[:, :, :, 5:], return_counts=True)

In [ ]:
x_train = x_train.astype(np.float32)
x_test = x_test.astype(np.float32)

In [ ]:
np.save(os.path.join(train_folder, 'train_images.npy'), x_train)
np.save(os.path.join(train_folder, 'train_labels.npy'), y_train)
np.save(os.path.join(test_folder, 'test_images.npy'), x_test)
np.save(os.path.join(test_folder, 'test_labels.npy'), y_test)